# 🧹 LMS Data Cleaning Notebook

**Objective:** Clean and transform the raw LMS training dataset for analysis.

**Steps:**
1. Load raw data & explore
2. Remove duplicates
3. Standardize text values
4. Handle missing values
5. Parse & validate dates
6. Create derived columns
7. Export cleaned dataset

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## 1. Load & Explore Raw Data

In [ ]:
df = pd.read_csv('../data/lms_raw.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
# Data types and info
df.info()

In [ ]:
# Check for data quality issues
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')
print(f'\n=== Unique Status Values ===')
print(df['status'].unique())
print(f'\n=== Unique Departments ===')
print(df['department'].unique())
print(f'\n=== Duplicates ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

## 2. Remove Duplicates

In [ ]:
initial_rows = len(df)
df = df.drop_duplicates()
print(f'Removed {initial_rows - len(df)} duplicate rows')
print(f'Remaining: {len(df)} rows')

## 3. Standardize Text Values

Fix inconsistent status and department names.

In [ ]:
# Status standardization
status_mapping = {
    'completed': 'Completed', 'COMPLETED': 'Completed', 'Completd': 'Completed',
    'pending': 'Pending', 'PENDING': 'Pending', 'Pendig': 'Pending',
}
df['status'] = df['status'].replace(status_mapping)
print(f'Status values after: {df["status"].unique()}')

# Department standardization
dept_mapping = {'I.T.': 'IT', 'Human Resources': 'HR', 'sales': 'Sales', 'marketing': 'Marketing'}
df['department'] = df['department'].replace(dept_mapping)
print(f'Departments after: {sorted(df["department"].dropna().unique())}')

## 4. Handle Missing Values

In [ ]:
# Fill missing departments with mode
mode_dept = df['department'].mode()[0]
missing_dept = df['department'].isnull().sum()
df['department'] = df['department'].fillna(mode_dept)
print(f'Filled {missing_dept} missing departments with mode: "{mode_dept}"')

# Fix Completed records without completion_date
mask = (df['status'] == 'Completed') & (df['completion_date'].isna() | (df['completion_date'] == ''))
df.loc[mask, 'status'] = 'Pending'
print(f'Fixed {mask.sum()} records: Completed without date → Pending')

## 5. Parse Date Columns

In [ ]:
df['assigned_date'] = pd.to_datetime(df['assigned_date'], format='mixed', dayfirst=True)
df['due_date'] = pd.to_datetime(df['due_date'], format='mixed', dayfirst=True)
df['completion_date'] = df['completion_date'].replace('', np.nan)
df['completion_date'] = pd.to_datetime(df['completion_date'], format='mixed', dayfirst=True, errors='coerce')

print(f'assigned_date dtype: {df["assigned_date"].dtype}')
print(f'due_date dtype: {df["due_date"].dtype}')
print(f'completion_date dtype: {df["completion_date"].dtype}')

## 6. Create Derived Columns

In [ ]:
# Completion time in days
completed_mask = df['status'] == 'Completed'
df['completion_time_days'] = np.nan
df.loc[completed_mask, 'completion_time_days'] = (
    df.loc[completed_mask, 'completion_date'] - df.loc[completed_mask, 'assigned_date']
).dt.days

# Is overdue flag
reference_date = pd.Timestamp('2026-03-31')
df['is_overdue'] = (df['status'] == 'Pending') & (df['due_date'] < reference_date)

# Assignment month
df['assigned_month'] = df['assigned_date'].dt.to_period('M').astype(str)

# Days allowed
df['days_allowed'] = (df['due_date'] - df['assigned_date']).dt.days

# On-time completion
df['completed_on_time'] = (df['status'] == 'Completed') & (df['completion_date'] <= df['due_date'])

print(f'Avg completion time: {df["completion_time_days"].mean():.1f} days')
print(f'Overdue records: {df["is_overdue"].sum()}')
print(f'On-time completions: {df["completed_on_time"].sum()}')

## 7. Final Check & Export

In [ ]:
print('=== Final Dataset Summary ===')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nMissing values:')
print(df.isnull().sum())
print(f'\nStatus distribution:')
print(df['status'].value_counts())
df.head()

In [ ]:
df.to_csv('../data/lms_cleaned.csv', index=False)
print('✅ Cleaned data saved to data/lms_cleaned.csv')